# R2AI2026 — sinh `pandas_query` trên Kaggle (T4 GPU)

Điều kiện trước khi chạy:
1. Bật **GPU T4** (Settings → Accelerator) và **Internet** (để `git clone` + tải model).
2. Upload `retrieval_results.jsonl` (build ở local bằng `python -m r2ai.retrieval.run_retrieval`) làm **Kaggle Dataset**, rồi Add Data vào notebook. File này đã nhúng sẵn CSV của các bảng candidate nên **không cần mount corpus 362MB**.
3. Sửa `REPO_URL` và `RETRIEVAL_PATH` bên dưới cho khớp.

Output: `predictions.jsonl` ghi **append + flush sau mỗi câu** trong `/kaggle/working` — tải về rồi chạy `python -m r2ai.packaging.assemble_submission` ở local (re-execute lại toàn bộ query trước khi đóng gói).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/r2ai-stage2.git"

# Đường dẫn dataset Kaggle đã Add Data vào notebook. Kaggle mount dataset tại
# `/kaggle/input/<dataset-slug>/`, KHÔNG phải theo URL `kaggle.com/datasets/<user>/<slug>` —
# nên nếu đặt cứng sai, mọi cell sau đều fail ở FileNotFoundError. Cell 3 sẽ tự dò lại bằng
# glob nếu đường dẫn này không tồn tại, nên chỉ cần sửa khi muốn ép dùng đúng 1 file.
RETRIEVAL_PATH = "/kaggle/input/r2ai2026/retrieval_results.jsonl"
PREDICTIONS_PATH = "/kaggle/working/predictions.jsonl"
PILOT_PREDICTIONS_PATH = "/kaggle/working/predictions_pilot.jsonl"  # file riêng, không đụng resume của full run
WORK_DIR = "/kaggle/working/exec"
PILOT_N = 20  # chạy thử trước khi chạy full 1.012 câu

In [ ]:
!git clone --depth 1 $REPO_URL /kaggle/working/r2ai-stage2
%cd /kaggle/working/r2ai-stage2
!pip install -q -r requirements.txt

In [ ]:
import logging
import os
import sys

sys.path.insert(0, "/kaggle/working/r2ai-stage2")
# Phải set TRƯỚC `import torch` để có tác dụng (CUDA context init lúc import).
# Đặt CẢ HAI tên: torch mới đã đổi sang `PYTORCH_ALLOC_CONF` (chính thông báo OOM thật cũng gợi ý
# tên mới này), tên cũ `PYTORCH_CUDA_ALLOC_CONF` giữ lại cho bản torch cũ hơn.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Tắt log INFO ồn (mỗi request HTTP khi tải model từ HuggingFace Hub, numexpr) — không phải lỗi,
# chỉ là noise. WARNING/ERROR vẫn hiện đầy đủ.
for name in ("httpx", "httpcore", "numexpr", "urllib3"):
    logging.getLogger(name).setLevel(logging.WARNING)

import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
      "| số GPU:", torch.cuda.device_count())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU 0: trống {free/2**30:.2f} GiB / tổng {total/2**30:.2f} GiB")
assert torch.cuda.is_available(), (
    "Không phát hiện GPU! Vào Settings (panel bên phải notebook) -> Accelerator -> chọn GPU T4 x2 "
    "(hoặc T4 x1) -> Save -> notebook sẽ restart session. Chạy lại từ đầu sau đó. Không có GPU thì "
    "load model 7B sẽ cực chậm/OOM trên CPU."
)

# Fail sớm ngay tại đây nếu cell cài đặt ở trên bị lỗi, thay vì để lỗi rơi xuống tận lúc load model 7B.
import bitsandbytes
import numpy
import pandas
import transformers

print("pandas", pandas.__version__, "| numpy", numpy.__version__,
      "| transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__)

# Tự dò file retrieval nếu đường dẫn đặt ở cell 1 không tồn tại (tên dataset khi Add Data
# thường khác với slug trên URL) — fail sớm ở đây kèm danh sách file thật, thay vì để lỗi
# FileNotFoundError rơi xuống giữa pilot.
import glob
import os.path

if not os.path.exists(RETRIEVAL_PATH):
    found = sorted(glob.glob("/kaggle/input/**/retrieval_results*.jsonl", recursive=True))
    print(f"Không thấy {RETRIEVAL_PATH}; tìm được: {found}")
    assert found, (
        "Chưa Add Data dataset chứa retrieval_results.jsonl vào notebook (panel phải -> Add Input). "
        f"Nội dung /kaggle/input hiện tại: {sorted(glob.glob('/kaggle/input/*'))}"
    )
    RETRIEVAL_PATH = found[0]

with open(RETRIEVAL_PATH, encoding="utf-8") as fh:
    n_questions = sum(1 for line in fh if line.strip())
print(f"retrieval: {RETRIEVAL_PATH} ({n_questions} câu)")
assert n_questions == 1012, f"Mong đợi 1.012 câu, file có {n_questions} — file retrieval có thể cũ/thiếu."

In [ ]:
# Đăng nhập HuggingFace Hub (không bắt buộc với Qwen, chỉ để bớt cảnh báo rate-limit khi tải model).
# Điền token thật vào đây SAU KHI đã import notebook này lên Kaggle — đừng commit bản đã điền token
# ngược lại về git. Bọc try/except: chạy "Save and Run All" không có người canh, token còn để
# placeholder hoặc sai thì KHÔNG được làm dừng cả notebook (Qwen tải được không cần token).
from huggingface_hub import login

try:
    login("HF_TOKEN")  # TODO: dán token HuggingFace thật vào đây trên Kaggle
    print("Đã đăng nhập HuggingFace Hub.")
except Exception as exc:
    print(f"Bỏ qua đăng nhập HF ({exc}) — vẫn chạy tiếp không token, chỉ bị giới hạn rate-limit.")

## 1. Smoke test: prompt + sandbox (không cần GPU)

`--dry-run` không nạp LLM: chỉ dựng prompt, ghi CSV ra đĩa và chạy sandbox. Bắt sớm lỗi encode CSV / đường dẫn trước khi tốn thời gian GPU.

In [ ]:
from r2ai.generation.run_generation import run

stats = run(
    retrieval_path=RETRIEVAL_PATH,
    out_path="/kaggle/working/predictions_dryrun.jsonl",
    work_dir=WORK_DIR,
    limit=3,
    dry_run=True,
    resume=False,
)
assert stats["exec_ok"] > 0, (
    f"Smoke test thất bại (exec_ok={stats['exec_ok']}/{stats['attempted']}) — xem traceback/lỗi phía "
    "trên. 'Save and Run All' dừng ở đây, không tốn thời gian chạy tiếp pilot/full khi plumbing hỏng."
)
print("smoke test OK:", stats)

# --- Kiểm tra prompt V2 thật sự có chú thích cột/đơn vị (bản sửa Bug 15) ---------------------------
# Gợi ý cột/đơn vị được tính lúc dựng prompt từ header nhúng trong `csv_text`. Nếu file retrieval
# upload lên Kaggle là bản CŨ (build trước khi sửa Bug 14) thì header chưa gộp nhiều tầng -> chú
# thích vẫn có nhưng chất lượng tụt hẳn mà KHÔNG có lỗi nào báo. In ra để mắt xác nhận trước khi
# tốn nhiều giờ GPU.
from pathlib import Path

from r2ai.config import load_config
from r2ai.generation.run_generation import load_retrieval_results
from r2ai.prompting.build_prompt import build_prompt

_cfg = load_config()
_results = load_retrieval_results(Path(RETRIEVAL_PATH))
_prompt = build_prompt(
    _results[0],
    max_tables=int(_cfg["retrieval"]["candidates_in_prompt"]),
    max_csv_chars=int(_cfg["retrieval"]["max_csv_chars"]),
    max_prompt_chars=int(_cfg["generation"]["max_prompt_chars"]),
)
assert "Cột theo kỳ:" in _prompt.user, "Prompt thiếu chú thích cột/kỳ — code không phải bản V2?"
for _line in _prompt.user.splitlines():
    if _line.startswith(("<!-- Cột theo kỳ", "<!-- Đổi đơn vị", "- Đơn vị đáp án")):
        print(_line[:200])

_merged = sum(
    1
    for r in _results[:200]
    for c in r.candidates[:6]
    if " | " in c.csv_text.split("\n", 1)[0]
)
print(f"header nhiều tầng đã gộp (mẫu 200 câu đầu): {_merged} bảng")
assert _merged > 0, (
    "Không thấy header nào được gộp nhiều tầng — file retrieval_results.jsonl trên Kaggle nhiều khả "
    "năng là bản CŨ (trước khi sửa Bug 14). Build lại ở local (`build_table_index` + `run_retrieval`) "
    "rồi upload lại dataset."
)

## 2. Load model (1 lần) + pilot 20 câu — đo wall-clock/câu

Model được load 1 lần ở đây và tái sử dụng cho cả pilot và bước full ở dưới (không load lại 2 lần). Ước lượng tổng thời gian cho 1.012 câu so với giới hạn session Kaggle (~9-12h). Nếu quá lâu: đổi sang `Qwen/Qwen2.5-Coder-3B-Instruct` (sửa `configs/baseline.yaml`, chạy lại từ cell load model) hoặc giảm `candidates_in_prompt`.

Chạy "Save and Run All": pilot thất bại hoặc `exec_ok=0` sẽ **dừng notebook tại đây** — mẫu lỗi thật (`exec_error`) được in ngay dưới để chẩn đoán, không cần chạy thêm cell nào khác.

In [ ]:
import json
import time

import torch

from r2ai.config import load_config
from r2ai.generation.run_generation import LocalLLM, run

gen_cfg = load_config()["generation"]
llm = LocalLLM(
    gen_cfg["model"],
    load_in_4bit=bool(gen_cfg["load_in_4bit"]),
    max_new_tokens=int(gen_cfg["max_new_tokens"]),
    temperature=float(gen_cfg["temperature"]),
)

start = time.time()
stats = run(
    llm=llm,
    retrieval_path=RETRIEVAL_PATH,
    out_path=PILOT_PREDICTIONS_PATH,
    work_dir=WORK_DIR,
    limit=PILOT_N,
    resume=False,  # file riêng cho pilot -> luôn chạy lại từ đầu, không dây với resume của full run
)
elapsed = time.time() - start

print(f"pilot: {stats}")
print(f"  OOM khi generate: {stats['generate_oom']} lần | phải thử lại prompt ngắn hơn: {stats['generate_retried']} lần")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"  GPU 0 sau pilot: trống {free/2**30:.2f} GiB / {total/2**30:.2f} GiB | attention = {llm.attn_implementation}")
per_q = elapsed / PILOT_N
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"{per_q:.1f}s/câu -> {per_q * 1012 / 3600:.1f}h nếu chạy TUẦN TỰ 1 GPU (cell 3b)")
if n_gpu >= 2:
    print(f"            -> ~{per_q * 1012 / 3600 / n_gpu:.1f}h nếu chạy SONG SONG {n_gpu} GPU (cell 3a) <- dùng cách này")
if stats["generate_oom"] > PILOT_N * 0.1:
    print(
        f"  ⚠️ OOM {stats['generate_oom']}/{PILOT_N} câu: mỗi lần OOM đốt 1 lần prefill vô ích rồi mới retry. "
        "Nếu muốn đổi thời gian lấy độ an toàn: hạ `generation.max_prompt_chars` trong configs/baseline.yaml "
        "16000 -> 14000 (trung bình 4,50 bảng/câu thay vì 5,31) hoặc -> 13000 (3,86 bảng/câu), rồi chạy lại cell này."
    )

rows = [json.loads(line) for line in open(PILOT_PREDICTIONS_PATH, encoding="utf-8") if line.strip()]
failed = [r for r in rows if not r["exec_ok"]]
for row in failed[:5]:
    print(f"--- id={row['id']} exec_error ---")
    print((row["exec_error"] or "")[:300])
    print("pandas_query (300 ký tự cuối):", row["pandas_query"][-300:])

# Kiểm mắt BẮT BUỘC trước khi tốn nhiều giờ GPU: model có thật sự dùng chú thích cột/đơn vị không?
used_col = sum(1 for r in rows if ".iloc[:," in r["pandas_query"].replace(" ", "") or "COL" in r["pandas_query"])
used_scale = sum(1 for r in rows if any(t in r["pandas_query"] for t in ("SCALE", "1e-", "1e6", "1e9", "/ 1_000_000", "/1000000", "/ 1000000")))
print(f"\npilot dùng chỉ số cột: {used_col}/{len(rows)} câu | có hệ số đổi đơn vị: {used_scale}/{len(rows)} câu")
print("--- pandas_query của 2 câu đầu (đọc tay xem có lấy đúng cột + đổi đơn vị) ---")
for row in rows[:2]:
    print(f"### id={row['id']} answer={row['answer']}")
    print(row["pandas_query"].split("pd.DataFrame.to_num")[-1][-700:])

assert stats["exec_ok"] > 0, (
    f"Pilot THẤT BẠI (exec_ok={stats['exec_ok']}/{stats['attempted']}) — xem exec_error in ở trên. "
    "'Save and Run All' dừng tại đây, không chạy tiếp bước full 1.012 câu (~nhiều giờ)."
)

## 3. Chạy full — CHỌN 1 trong 2 cách

**3a (khuyến nghị khi session có T4 x2): chạy SONG SONG 2 GPU** — cell dưới cùng. Mỗi GPU chạy 1 process
riêng với 1 bản model riêng (model 7B 4-bit chỉ ~4,5GB, thừa chỗ trên 1 T4 16GB), mỗi process nhận một nửa
câu hỏi (`--shard 0/2` và `--shard 1/2`, chia theo `id % 2`). Wall-clock giảm ~2 lần. **Phải giải phóng
model đang giữ trong notebook trước** (cell 3a tự làm) vì nó đang chiếm GPU 0.

**3b: chạy tuần tự 1 GPU** (cách cũ, `resume=True` nên chạy lại là tiếp tục từ chỗ dừng). Chỉ dùng khi
session chỉ có 1 GPU, hoặc muốn dùng lại `llm` đã load ở cell pilot.

> Ước lượng thời gian lấy từ pilot: `Xs/câu × 1012`. Nếu vượt giới hạn session (~12h) thì bắt buộc dùng 3a,
> hoặc đổi `configs/baseline.yaml` sang `Qwen/Qwen2.5-Coder-3B-Instruct` (nhanh ~2 lần, chất lượng thấp hơn).

In [ ]:
# === 3a. CHẠY SONG SONG 2 GPU (khuyến nghị khi session có T4 x2) ===================================
# Mỗi GPU = 1 process = 1 bản model riêng, mỗi process làm một nửa câu hỏi (id % 2). Không dùng
# `device_map="auto"` để chia 1 model ra 2 GPU: cách đó KHÔNG nhanh hơn (decode vẫn tuần tự qua từng
# layer, thêm chi phí chuyển tensor giữa 2 GPU) và chính là nguyên nhân OOM ở CẬP NHẬT 9.
import os
import subprocess
import sys
import time

import torch

# Giải phóng model đang giữ trong notebook (cell pilot) — nếu không, GPU 0 mất ~5GB và process con OOM.
for _name in ("llm",):
    if _name in globals():
        del globals()[_name]
import gc

gc.collect()
torch.cuda.empty_cache()
n_gpu = torch.cuda.device_count()
print(f"số GPU: {n_gpu}")
assert n_gpu >= 2, (
    f"Chỉ thấy {n_gpu} GPU — cell này cần T4 x2 (Settings -> Accelerator -> GPU T4 x2). "
    "Với 1 GPU thì dùng cell 3b (tuần tự)."
)

SHARD_PATHS = [f"/kaggle/working/predictions_shard{i}.jsonl" for i in range(n_gpu)]
procs = []
for i in range(n_gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))  # process i chỉ thấy GPU i (thành "GPU 0" của nó)
    log = open(f"/kaggle/working/shard{i}.log", "w")
    procs.append(
        (
            subprocess.Popen(
                [
                    sys.executable, "-m", "r2ai.generation.run_generation",
                    "--retrieval", str(RETRIEVAL_PATH),
                    "--out", SHARD_PATHS[i],
                    "--work-dir", f"/kaggle/working/exec_shard{i}",
                    "--shard", f"{i}/{n_gpu}",
                ],
                cwd="/kaggle/working/r2ai-stage2",
                env=env,
                stdout=log,
                stderr=subprocess.STDOUT,
            ),
            log,
        )
    )
    print(f"đã khởi động shard {i}/{n_gpu} trên GPU {i} -> {SHARD_PATHS[i]}")

# Theo dõi tiến độ: đếm số dòng đã ghi trong từng file predictions (append+flush sau mỗi câu).
start = time.time()
while any(p.poll() is None for p, _ in procs):
    time.sleep(120)
    counts = [sum(1 for _ in open(p, encoding="utf-8")) if os.path.exists(p) else 0 for p in SHARD_PATHS]
    done = sum(counts)
    speed = done / max(time.time() - start, 1)
    eta = (1012 - done) / speed / 3600 if speed > 0 else float("inf")
    print(f"[{(time.time()-start)/60:6.1f} phút] {counts} = {done}/1012 câu | còn ~{eta:.1f}h", flush=True)

for i, (p, log) in enumerate(procs):
    log.close()
    print(f"shard {i} kết thúc, returncode={p.returncode}")
    print(open(f"/kaggle/working/shard{i}.log", encoding="utf-8").read()[-1500:])


In [ ]:
# Gộp các file shard thành 1 `predictions.jsonl` duy nhất để tải về (bước assemble ở local cần 1 file).
import json

rows = {}
for path in SHARD_PATHS:
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                row = json.loads(line)
                rows[row["id"]] = row  # dòng sau đè dòng trước cùng id (lần chạy lại là bản mới hơn)

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as out:
    for qid in sorted(rows):
        out.write(json.dumps(rows[qid], ensure_ascii=False) + "\n")

ok = sum(1 for r in rows.values() if r["exec_ok"])
print(f"gộp {len(rows)}/1012 câu -> {PREDICTIONS_PATH} | exec_ok={ok} ({ok/max(len(rows),1)*100:.1f}%)")
missing = sorted(set(range(1, 1013)) - set(rows))
print(f"thiếu {len(missing)} câu" + (f": {missing[:20]}..." if missing else " (đủ)"))
for row in [r for r in rows.values() if not r["exec_ok"]][:5]:
    print(row["id"], "->", (row["exec_error"] or "")[:200])


In [ ]:
# === 3b. Chạy TUẦN TỰ 1 GPU (chỉ dùng khi session có 1 GPU, hoặc 3a lỗi) ===========================
# `resume=True` (mặc định): chạy lại cell này sau khi session bị ngắt là tiếp tục từ chỗ dừng.
# ĐỪNG chạy cell này sau cell 3a — hai bên ghi vào cùng `PREDICTIONS_PATH` (3a sẽ bị ghi đè lẫn lộn).
from r2ai.config import load_config
from r2ai.generation.run_generation import LocalLLM, run

if "llm" not in globals():  # cell 3a đã xoá `llm` để nhường GPU cho process con -> load lại
    _gen_cfg = load_config()["generation"]
    llm = LocalLLM(
        _gen_cfg["model"],
        load_in_4bit=bool(_gen_cfg["load_in_4bit"]),
        max_new_tokens=int(_gen_cfg["max_new_tokens"]),
        temperature=float(_gen_cfg["temperature"]),
    )

stats = run(llm=llm, retrieval_path=RETRIEVAL_PATH, out_path=PREDICTIONS_PATH, work_dir=WORK_DIR)
print(f"full run: {stats}")

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in open(PREDICTIONS_PATH, encoding="utf-8") if line.strip()]
print("tổng:", len(rows), "| id duy nhất:", len({r["id"] for r in rows}))
print("exec_ok:", Counter(r["exec_ok"] for r in rows))
for row in [r for r in rows if not r["exec_ok"]][:5]:
    print(row["id"], "->", (row["exec_error"] or "")[:200])

Tải `predictions.jsonl` về máy local, đặt vào `data/interim/`, rồi:

```bash
python -m r2ai.packaging.assemble_submission   # join + re-execute ở local
python -m r2ai.packaging.zip_submission        # validate + zip
```